=============================================================================
COMP64702 - Notebook 05 EXTENSION
Additional Evaluation — Generation Baseline + Faithfulness
=============================================================================
Add these cells to the END of your 05_evaluation.ipynb

What this adds:
  1. Generation baseline (LLM with NO context) vs our full RAG system
     → Statistical significance test on ROUGE-L and BERTScore
     → This is what gets you the 2 "competitive generation" marks

  2. Faithfulness scoring — does the answer stay grounded in retrieved chunks?
     → Key RAG-specific metric that shows evaluator sophistication

  3. Combined final report with ALL results in one place
=============================================================================


─────────────────────────────────────────────────────────────────────────────
CELL A — Load everything needed (run this first if in a fresh session)
─────────────────────────────────────────────────────────────────────────────


In [1]:

import os
import json
import pickle
import numpy as np
import torch
import faiss
from datetime import datetime
from scipy import stats
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from dotenv import load_dotenv
from huggingface_hub import login

os.chdir(r"D:\text mining\rag project")
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

# Load existing train outputs from notebook 04
with open("outputs/train_outputs.json", encoding="utf-8") as f:
    train_outputs = json.load(f)

with open("data/benchmark/train_set.json", encoding="utf-8") as f:
    train_set = json.load(f)

print(f"Loaded {len(train_outputs)} train outputs")
print(f"Loaded {len(train_set)} train benchmark pairs")

# Load LLM
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
device     = "cuda" if torch.cuda.is_available() else "cpu"
llm        = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
llm.eval()
print(f"LLM loaded on {device}")




Loaded 92 train outputs
Loaded 92 train benchmark pairs


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


LLM loaded on cpu


─────────────────────────────────────────────────────────────────────────────
CELL B — Generation baseline: LLM with NO retrieved context

This is the critical comparison for the 2 "competitive generation" marks.
We run Qwen on the same questions but WITHOUT any retrieved chunks.
Then we show our full RAG system beats this baseline significantly.
─────────────────────────────────────────────────────────────────────────────


In [6]:
NO_CONTEXT_SYSTEM = """You are a knowledgeable culinary assistant specialising in East Asian cuisine.
Answer the question as best you can from your own knowledge.
Keep your answer between 2 and 5 sentences."""
 
def generate_no_context(query, max_new_tokens=200):
    """
    Generates an answer using the LLM alone — no retrieved context.
    This is the generation baseline (RAG-free).
    """
    messages = [
        {"role": "system", "content": NO_CONTEXT_SYSTEM},
        {"role": "user",   "content": f"Question: {query}\n\nAnswer:"},
    ]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
 
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()
 
 
# Run no-context baseline on all train queries
print(f"\nRunning no-context generation baseline on {len(train_set)} queries...")
print("(This takes a few minutes)\n")
 
baseline_gen_outputs = []
 
for i, qa in enumerate(train_set, 1):
    answer = generate_no_context(qa["question"])
    baseline_gen_outputs.append({
        "id":           qa.get("id", f"Q{i}"),
        "question":     qa["question"],
        "gold_answer":  qa["answer"],
        "pred_answer":  answer,
        "type":         qa.get("type", "unknown"),
        "difficulty":   qa.get("difficulty", "unknown"),
    })
    if i % 10 == 0:
        print(f"  {i}/{len(train_set)} done")
 
# Save baseline outputs
with open("outputs/evaluation/baseline_gen_outputs.json", "w", encoding="utf-8") as f:
    json.dump(baseline_gen_outputs, f, ensure_ascii=False, indent=2)
 
print(f"\nBaseline generation outputs saved")
 


Running no-context generation baseline on 92 queries...
(This takes a few minutes)

  10/92 done
  20/92 done
  30/92 done
  40/92 done
  50/92 done
  60/92 done
  70/92 done
  80/92 done
  90/92 done

Baseline generation outputs saved



    Generates an answer using the LLM alone — no retrieved context.
    This is the generation baseline (RAG-free).
    


─────────────────────────────────────────────────────────────────────────────
CELL C — Compute ROUGE and BERTScore for both systems
─────────────────────────────────────────────────────────────────────────────


In [7]:

def compute_rouge_scores(outputs):
    scorer_  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for o in outputs:
        result = scorer_.score(o["gold_answer"], o["pred_answer"])
        r1.append(result["rouge1"].fmeasure)
        r2.append(result["rouge2"].fmeasure)
        rl.append(result["rougeL"].fmeasure)
    return {"rouge1": r1, "rouge2": r2, "rougeL": rl}


def compute_bert_scores(outputs):
    preds = [o["pred_answer"]  for o in outputs]
    refs  = [o["gold_answer"]  for o in outputs]
    print("  Computing BERTScore...")
    _, _, F1 = bert_score_fn(preds, refs, model_type="distilbert-base-uncased",
                              lang="en", verbose=False)
    return F1.numpy().tolist()


print("Computing metrics for RAG system...")
rag_rouge  = compute_rouge_scores(train_outputs)
rag_bert   = compute_bert_scores(train_outputs)

print("Computing metrics for no-context baseline...")
base_rouge = compute_rouge_scores(baseline_gen_outputs)
base_bert  = compute_bert_scores(baseline_gen_outputs)




Computing metrics for RAG system...
  Computing BERTScore...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\Rohan\anaconda3\envs\rag-project\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohan\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing metrics for no-context baseline...
  Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


─────────────────────────────────────────────────────────────────────────────
CELL D — Statistical significance tests for generation
Paired t-test on ROUGE-L and BERTScore
Both need to be significant for full marks
─────────────────────────────────────────────────────────────────────────────


Runs a paired t-test and prints results clearly.


In [8]:
def paired_ttest(system_scores, baseline_scores, metric_name):
    """Runs a paired t-test and prints results clearly."""
    # Match lengths
    n = min(len(system_scores), len(baseline_scores))
    s = system_scores[:n]
    b = baseline_scores[:n]
 
    t_stat, p_value = stats.ttest_rel(s, b)
    sig = p_value < 0.05
 
    print(f"\n  {metric_name}")
    print(f"    RAG system mean : {np.mean(s):.4f}")
    print(f"    Baseline mean   : {np.mean(b):.4f}")
    print(f"    Difference      : {np.mean(s) - np.mean(b):+.4f}")
    print(f"    t-statistic     : {t_stat:.4f}")
    print(f"    p-value         : {p_value:.4f}")
    print(f"    Significant     : {'YES (p < 0.05)' if sig else 'NO (p >= 0.05)'}")
 
    return {
        "rag_mean":   float(np.mean(s)),
        "base_mean":  float(np.mean(b)),
        "difference": float(np.mean(s) - np.mean(b)),
        "t_stat":     float(t_stat),
        "p_value":    float(p_value),
        "significant":bool(sig),
    }
 
 
print("="*60)
print("  GENERATION SIGNIFICANCE TESTS: RAG vs No-Context Baseline")
print("="*60)
 
rougeL_sig  = paired_ttest(rag_rouge["rougeL"], base_rouge["rougeL"], "ROUGE-L")
rouge1_sig  = paired_ttest(rag_rouge["rouge1"], base_rouge["rouge1"], "ROUGE-1")
bert_sig    = paired_ttest(rag_bert,            base_bert,            "BERTScore F1")
 

  GENERATION SIGNIFICANCE TESTS: RAG vs No-Context Baseline

  ROUGE-L
    RAG system mean : 0.3668
    Baseline mean   : 0.1543
    Difference      : +0.2125
    t-statistic     : 8.5679
    p-value         : 0.0000
    Significant     : YES (p < 0.05)

  ROUGE-1
    RAG system mean : 0.4103
    Baseline mean   : 0.2145
    Difference      : +0.1958
    t-statistic     : 7.8416
    p-value         : 0.0000
    Significant     : YES (p < 0.05)

  BERTScore F1
    RAG system mean : 0.8425
    Baseline mean   : 0.7815
    Difference      : +0.0610
    t-statistic     : 8.2355
    p-value         : 0.0000
    Significant     : YES (p < 0.05)


─────────────────────────────────────────────────────────────────────────────
CELL E — Faithfulness scoring
Does the predicted answer stay grounded in the retrieved chunks?
This is an advanced RAG-specific metric that shows evaluator sophistication
─────────────────────────────────────────────────────────────────────────────


In [9]:
def compute_faithfulness(train_outputs, threshold=0.25):
    """
    Faithfulness: measures whether the predicted answer is grounded in
    the retrieved context chunks.
 
    Method: token-level recall of answer words that appear in the context.
    A faithful answer uses words from the context rather than hallucinating.
 
    This is a lightweight proxy for full LLM-judge faithfulness scoring.
    threshold: minimum fraction of answer words that must appear in context
    """
    import re
 
    def normalise(text):
        return set(re.sub(r'[^\w\s]', '', text.lower()).split())
 
    faithful_count = 0
    faithfulness_scores = []
 
    for output in train_outputs:
        pred_words    = normalise(output.get("pred_answer", ""))
        context_words = set()
 
        # Collect all words from retrieved chunks
        for chunk in output.get("retrieved", []):
            context_words.update(normalise(chunk.get("text", "")))
 
        if not pred_words or not context_words:
            faithfulness_scores.append(0.0)
            continue
 
        # What fraction of predicted answer words appear in the context?
        overlap      = pred_words & context_words
        faithfulness = len(overlap) / len(pred_words)
        faithfulness_scores.append(faithfulness)
 
        if faithfulness >= threshold:
            faithful_count += 1
 
    mean_faith = float(np.mean(faithfulness_scores))
    faith_rate = faithful_count / len(faithfulness_scores) if faithfulness_scores else 0
 
    return faithfulness_scores, mean_faith, faith_rate
 
 
faith_scores, faith_mean, faith_rate = compute_faithfulness(train_outputs)
 
print(f"\nFAITHFULNESS EVALUATION")
print(f"  Mean faithfulness score : {faith_mean:.4f}")
print(f"  Faithful answers (>25%) : {faith_rate:.4f} ({faith_rate*100:.1f}%)")
print(f"  Min score               : {min(faith_scores):.4f}")
print(f"  Max score               : {max(faith_scores):.4f}")
 


FAITHFULNESS EVALUATION
  Mean faithfulness score : 0.4804
  Faithful answers (>25%) : 0.9565 (95.7%)
  Min score               : 0.1724
  Max score               : 1.0000



    Faithfulness: measures whether the predicted answer is grounded in
    the retrieved context chunks.

    Method: token-level recall of answer words that appear in the context.
    A faithful answer uses words from the context rather than hallucinating.

    This is a lightweight proxy for full LLM-judge faithfulness scoring.
    threshold: minimum fraction of answer words that must appear in context
    


─────────────────────────────────────────────────────────────────────────────
CELL F — Complete comparison table (all metrics, both systems)
Copy this directly into your presentation
─────────────────────────────────────────────────────────────────────────────


In [10]:
print("\n" + "="*65)
print("  COMPLETE EVALUATION RESULTS")
print("="*65)
 
print(f"\n  GENERATION METRICS — RAG System vs No-Context Baseline")
print(f"  {'Metric':<20} {'Baseline':>10} {'RAG System':>12} {'Δ':>8} {'Sig?':>6}")
print("  " + "─"*58)
 
gen_comparisons = [
    ("ROUGE-1",     np.mean(base_rouge["rouge1"]), np.mean(rag_rouge["rouge1"])),
    ("ROUGE-2",     np.mean(base_rouge["rouge2"]), np.mean(rag_rouge["rouge2"])),
    ("ROUGE-L",     np.mean(base_rouge["rougeL"]), np.mean(rag_rouge["rougeL"])),
    ("BERTScore F1",np.mean(base_bert),            np.mean(rag_bert)),
]
 
sig_results = {
    "ROUGE-L":      rougeL_sig["significant"],
    "BERTScore F1": bert_sig["significant"],
}
 
for metric, base_val, rag_val in gen_comparisons:
    delta = rag_val - base_val
    sign  = "+" if delta >= 0 else ""
    sig   = sig_results.get(metric, "")
    sig_str = "YES" if sig == True else ("NO" if sig == False else "")
    print(f"  {metric:<20} {base_val:>10.4f} {rag_val:>12.4f} {sign}{delta:>7.4f} {sig_str:>6}")
 
print(f"\n  FAITHFULNESS (RAG system only)")
print(f"  Mean faithfulness score : {faith_mean:.4f}")
print(f"  Faithful answer rate    : {faith_rate*100:.1f}%")
 
print(f"\n  KEY FINDING:")
rougeL_improved = np.mean(rag_rouge["rougeL"]) > np.mean(base_rouge["rougeL"])
bert_improved   = np.mean(rag_bert) > np.mean(base_bert)
if rougeL_improved and bert_improved:
    print(f"  Our RAG system outperforms the no-context baseline on all")
    print(f"  generation metrics, confirming that retrieved context")
    print(f"  meaningfully improves answer quality.")
else:
    print(f"  Note: If baseline outperforms RAG on some metrics, this is")
    print(f"  expected with a 0.5B model — the context window can confuse")
    print(f"  small models. Mention this in your presentation as a known")
    print(f"  limitation and discuss how a larger model would benefit more.")
 
print("="*65)
 


  COMPLETE EVALUATION RESULTS

  GENERATION METRICS — RAG System vs No-Context Baseline
  Metric                 Baseline   RAG System        Δ   Sig?
  ──────────────────────────────────────────────────────────
  ROUGE-1                  0.2145       0.4103 + 0.1958       
  ROUGE-2                  0.0556       0.2522 + 0.1966       
  ROUGE-L                  0.1543       0.3668 + 0.2125    YES
  BERTScore F1             0.7815       0.8425 + 0.0610    YES

  FAITHFULNESS (RAG system only)
  Mean faithfulness score : 0.4804
  Faithful answer rate    : 95.7%

  KEY FINDING:
  Our RAG system outperforms the no-context baseline on all
  generation metrics, confirming that retrieved context
  meaningfully improves answer quality.


─────────────────────────────────────────────────────────────────────────────
CELL G — Save extended evaluation report
─────────────────────────────────────────────────────────────────────────────


In [11]:

# Load the existing report if it exists
report_path = "outputs/evaluation/evaluation_report.json"
try:
    with open(report_path, encoding="utf-8") as f:
        full_report = json.load(f)
except FileNotFoundError:
    full_report = {}

# Extend with generation baseline comparison
full_report["generation_baseline_comparison"] = {
    "system":   "no-context (LLM only, no retrieval)",
    "metrics": {
        "rouge1": {
            "rag":      float(np.mean(rag_rouge["rouge1"])),
            "baseline": float(np.mean(base_rouge["rouge1"])),
        },
        "rouge2": {
            "rag":      float(np.mean(rag_rouge["rouge2"])),
            "baseline": float(np.mean(base_rouge["rouge2"])),
        },
        "rougeL": {
            "rag":      float(np.mean(rag_rouge["rougeL"])),
            "baseline": float(np.mean(base_rouge["rougeL"])),
            "significance": rougeL_sig,
        },
        "bertscore": {
            "rag":      float(np.mean(rag_bert)),
            "baseline": float(np.mean(base_bert)),
            "significance": bert_sig,
        },
    },
}

full_report["faithfulness"] = {
    "mean_score":    faith_mean,
    "faithful_rate": faith_rate,
    "threshold":     0.25,
    "method":        "token overlap between answer and retrieved context",
}

full_report["updated_at"] = datetime.now().isoformat()

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(full_report, f, ensure_ascii=False, indent=2)

print(f"\nExtended evaluation report saved -> {report_path}")
print("\nEvaluation complete. All results are ready for your presentation.")




Extended evaluation report saved -> outputs/evaluation/evaluation_report.json

Evaluation complete. All results are ready for your presentation.
